In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchattacks
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.001
EPSILON = 0.1 
print(f"🖥️ Device: {device}")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
testloader  = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class Model_A(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 64, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2)
        )
        with torch.no_grad():
            self.feat_size = self.features(torch.zeros(1, 1, 28, 28)).numel()
        self.fc1 = nn.Linear(self.feat_size, 256)
        self.fc2 = nn.Linear(256, 10)
        self.relu = nn.ReLU()
        self.feature_maps = {}

    def forward(self, x):
        x = self.features(x)
        self.feature_maps['layer2'] = x 
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_clean(model, loader, epochs, lr):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        correct, total, loss_sum = 0, 0, 0.0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            _, pred = outputs.max(1)
            correct += pred.eq(y).sum().item()
            total += y.size(0)
            loss_sum += loss.item()
        acc = 100 * correct / total
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss_sum/len(loader):.4f} | Acc: {acc:.2f}%")
    return model

def extract_features_and_labels(model, loader):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            _ = model(X) 
            f = model.feature_maps['layer2'].cpu().numpy().reshape(X.size(0), -1)
            feats.append(f)
            labels.append(y.numpy().astype(int))
    return np.vstack(feats), np.concatenate(labels)

class PCADetector:
    def __init__(self, n_components=0.95):
        self.pca = PCA(n_components=n_components, svd_solver='full')
    def fit(self, X):
        self.pca.fit(X)
        return self
    def score(self, X):
        
        X_red = self.pca.transform(X)
        X_rec = self.pca.inverse_transform(X_red)
        return np.sum((X - X_rec) ** 2, axis=1)

class OCSVMDetector:
    def __init__(self, nu=0.01):
        self.clf = OneClassSVM(nu=nu, gamma='scale')
    def fit(self, X):
        self.clf.fit(X)
        return self
    def score(self, X):
        # decision_function: чем меньше, тем более аномально. Инвертируем знак.
        return -self.clf.decision_function(X)

class IFDetector:
    def __init__(self, n_estimators=100):
        self.clf = IsolationForest(n_estimators=n_estimators, contamination='auto', random_state=42, n_jobs=-1)
    def fit(self, X):
        self.clf.fit(X)
        return self
    def score(self, X):
        
        return -self.clf.decision_function(X)

def tpr_at_fpr(y_true, y_scores, fpr_target):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    return np.interp(fpr_target, fpr, tpr)

def evaluate_attack(model, loader, detector, attack_fn, attack_name):
    model.eval()
    X_adv_list, y_adv_list = [], []
    
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        
        X_adv = attack_fn(X, y)
        
        with torch.no_grad():
            _ = model(X_adv)
            f = model.feature_maps['layer2'].cpu().numpy().reshape(X_adv.size(0), -1)
        X_adv_list.append(f)
        y_adv_list.append(y.cpu().numpy().astype(int))
        
    X_adv = np.vstack(X_adv_list)
    y_adv = np.concatenate(y_adv_list)
    
    scores_adv = detector.score(X_adv)
    return scores_adv, y_adv

if __name__ == "__main__":
    print(f"\n🚀 Обучение Model_A (чистое)...")
    model = Model_A().to(device)
    train_clean(model, trainloader, EPOCHS, LR)

    print(f"\n🔍 Извлечение чистых активаций и обучение детекторов...")
    X_clean, y_clean = extract_features_and_labels(model, testloader)
    
    detectors = {
        "OCSVM": OCSVMDetector(nu=0.01),
        "IF": IFDetector(n_estimators=100),
        "PCA": PCADetector(n_components=0.95)
    }
    
    scores_clean = {}
    for name, det in detectors.items():
        det.fit(X_clean)
        scores_clean[name] = det.score(X_clean)

    print(f"\n📈 Сравнение эффективности обнаружения атак (ε = {EPSILON}):")
    header = f"{'Детектор':<8} | {'Атака':<6} | {'AUC':<7} | {'TPR@FPR=1%':<12}"
    print(header)
    print("-" * len(header))

    attacks = {
        "FGSM": torchattacks.FGSM(model, eps=EPSILON),
        "PGD": torchattacks.PGD(model, eps=EPSILON, steps=10, alpha=EPSILON/4, random_start=False)
    }

    results = {}
    
    for det_name, detector in detectors.items():
        for atk_name, atk_fn in attacks.items():
            scores_adv, y_adv = evaluate_attack(model, testloader, detector, atk_fn, atk_name)
            
            sc_c = scores_clean[det_name].copy()
            sc_a = scores_adv.copy()
            if np.mean(sc_c) > np.mean(sc_a):
                sc_c = -sc_c
                sc_a = -sc_a
            
            y_true = np.concatenate([np.zeros(len(sc_c)), np.ones(len(sc_a))])
            y_scores = np.concatenate([sc_c, sc_a])
            
            auc = roc_auc_score(y_true, y_scores)
            tpr1 = tpr_at_fpr(y_true, y_scores, 0.01)
            
            print(f"{det_name:<8} | {atk_name:<6} | {auc:<7.4f} | {tpr1:<12.3f}")
            results[(det_name, atk_name)] = (auc, tpr1)

    print(f"\n📊 Сводная таблица (для вставки в работу):")
    print(f"{'Детектор':<8} | {'FGSM (AUC)':<12} | {'PGD (AUC)':<12} | {'FGSM (TPR@1%)':<15} | {'PGD (TPR@1%)':<15}")
    print("-" * 75)
    for det_name in ["OCSVM", "IF", "PCA"]:
        auc_fgsm, tpr_fgsm = results[(det_name, "FGSM")]
        auc_pgd, tpr_pgd = results[(det_name, "PGD")]
        print(f"{det_name:<8} | {auc_fgsm:<12.4f} | {auc_pgd:<12.4f} | {tpr_fgsm:<15.3f} | {tpr_pgd:<15.3f}")

🖥️ Device: cpu

🚀 Обучение Model_A (чистое)...
Epoch 1/10 | Loss: 0.4253 | Acc: 84.45%
Epoch 2/10 | Loss: 0.2716 | Acc: 90.07%
Epoch 3/10 | Loss: 0.2265 | Acc: 91.67%
Epoch 4/10 | Loss: 0.1892 | Acc: 93.01%
Epoch 5/10 | Loss: 0.1626 | Acc: 93.96%
Epoch 6/10 | Loss: 0.1343 | Acc: 95.04%
Epoch 7/10 | Loss: 0.1130 | Acc: 95.76%
Epoch 8/10 | Loss: 0.0926 | Acc: 96.56%
Epoch 9/10 | Loss: 0.0752 | Acc: 97.27%
Epoch 10/10 | Loss: 0.0589 | Acc: 97.82%

🔍 Извлечение чистых активаций и обучение детекторов...

📈 Сравнение эффективности обнаружения атак (ε = 0.1):
Детектор | Атака  | AUC     | TPR@FPR=1%  
------------------------------------------
OCSVM    | FGSM   | 0.9757  | 0.687       
OCSVM    | PGD    | 0.9720  | 0.659       
IF       | FGSM   | 0.5976  | 0.008       
IF       | PGD    | 0.5751  | 0.005       
PCA      | FGSM   | 0.8435  | 0.284       
PCA      | PGD    | 0.8498  | 0.306       

📊 Сводная таблица (для вставки в работу):
Детектор | FGSM (AUC)   | PGD (AUC)    | FGSM (TPR@1%)